# E2E Type Progression

This notebook illustrates the typed interface progression used by `mellea-lrc`:

1. `Path` -> `PreprocessedDocument`
2. `PreprocessedDocument` -> `ExtractedDocument`
3. `ExtractedDocument` -> `InferredDocument`
4. `InferredDocument` -> `RetrievedDocument`
5. `RetrievedDocument` -> `AssessedDocument`

Each step writes `local/snapshots/<doc>/<stage>.json`, reloads through `deserialize_*`, and reads its input from the previous snapshot. Re-run any step after setup as long as upstream snapshots exist.

In [ ]:
from pathlib import Path

from IPython.display import display

# Always process the bookmark and the first ten numbered fixtures.
TEST_DATA_DIR = Path("../local/test_data")
BOOKMARKED_PATH = Path("../local/bookmarked/bookmarked.txt")
SNAPSHOT_ROOT = Path("../local/snapshots")
MELLEA_CONCURRENCY = 5

# Run everything up to and including this phase. Options: preprocessed, extraction, inferred, retrieval, assessment
PHASE: str = "assessment"
SNAPSHOT_STAGES = ("preprocessed", "extraction", "inferred", "retrieval", "assessment")
_phase_idx = SNAPSHOT_STAGES.index(PHASE)


def _should_run(stage: str) -> bool:
    return SNAPSHOT_STAGES.index(stage) <= _phase_idx

_all_paths = [BOOKMARKED_PATH] + [TEST_DATA_DIR / f"{index}.txt" for index in range(1, 3)]
DOC_PATHS = [
    path for path in _all_paths
    if path.is_file() and path.read_text(encoding="utf-8").strip()
]
skipped = [path for path in _all_paths if not path.is_file() or not path.read_text(encoding="utf-8").strip()]
if skipped:
    display({"skipped (not found)": [str(p) for p in skipped]})

display({"documents": [str(path) for path in DOC_PATHS]})

{'skipped (not found)': ['../local/bookmarked/bookmarked.txt']}

{'documents': ['../local/test_data/1.txt', '../local/test_data/2.txt']}

In [2]:
from __future__ import annotations

import json
import os
import shutil
from pathlib import Path


from mellea_lrc.core.env import load_env_file

load_env_file(Path("../.env"), override=True)

from mellea_lrc.assessment import (
    CitationAssessmentResult,
    MelleaCallContext,
    run_assessment_async,
)
from mellea_lrc.extraction import run_extraction
from mellea_lrc.jurisdiction_inference import infer_jurisdiction
from mellea_lrc.llm import llm_api_config_from_env
from mellea_lrc.preprocessing import run_preprocessing
from mellea_lrc.serialization import (
    deserialize_assessed_document,
    deserialize_extracted_document,
    deserialize_inferred_document,
    deserialize_preprocessed_document,
    deserialize_retrieved_document,
    serialize_assessed_document,
    serialize_extracted_document,
    serialize_inferred_document,
    serialize_preprocessed_document,
    serialize_retrieved_document,
)
from mellea_lrc.retrieval import run_retrieval



from collections.abc import Callable

def reset_snapshot_dir(doc_path: Path) -> Path:
    snapshot_dir = SNAPSHOT_ROOT / doc_path.stem
    shutil.rmtree(snapshot_dir, ignore_errors=True)
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    return snapshot_dir


def snapshot_path(doc_path: Path, stage: str) -> Path:
    if stage not in SNAPSHOT_STAGES:
        msg = f"Unknown snapshot stage: {stage}"
        raise ValueError(msg)
    snapshot_dir = SNAPSHOT_ROOT / doc_path.stem
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    return snapshot_dir / f"{stage}.json"


def write_snapshot(doc_path: Path, stage: str, payload: object) -> Path:
    path = snapshot_path(doc_path, stage)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return path


def load_snapshot(
    doc_path: Path, stage: str, deserialize: Callable[[dict[str, object]], object]
) -> object:
    path = snapshot_path(doc_path, stage)
    if not path.exists():
        msg = f"Missing snapshot: {path}"
        raise FileNotFoundError(msg)
    payload = json.loads(path.read_text(encoding="utf-8"))
    return deserialize(payload)


def summarize_snapshot(stage: str, path: Path, payload: dict[str, object]) -> dict[str, object]:
    summary = {
        "stage": stage,
        "snapshot": str(path),
        "top_level_keys": sorted(payload.keys()),
    }
    counts = payload.get("counts")
    if isinstance(counts, dict):
        summary["counts"] = counts
    return summary

## Step 1: Preprocess

`run_preprocessing(Path)` produces the first typed boundary object: `PreprocessedDocument`. The snapshot is reloaded with `deserialize_preprocessed_document` before extraction.

In [3]:
if not _should_run("preprocessed"):
    display("Skipping: phase {PHASE!r} < preprocessed")
else:
    for doc_path in DOC_PATHS:
        reset_snapshot_dir(doc_path)
        preprocessed_raw = run_preprocessing(doc_path)
        preprocessed_payload = serialize_preprocessed_document(preprocessed_raw)
        preprocessed_path = write_snapshot(doc_path, "preprocessed", preprocessed_payload)
        preprocessed = load_snapshot(doc_path, "preprocessed", deserialize_preprocessed_document)
        display(
            {
                "document": doc_path.name,
                **summarize_snapshot("preprocessed", preprocessed_path, preprocessed_payload),
                "chars": len(preprocessed.text),
                "source_path": preprocessed.source_metadata.path,
                "source_format": preprocessed.source_metadata.format.value,
                "backend": preprocessed.preprocessing_metadata.backend.value,
            }
        )

{'document': '1.txt',
 'stage': 'preprocessed',
 'snapshot': '../local/snapshots/1/preprocessed.json',
 'top_level_keys': ['artifact_type',
  'preprocessing_metadata',
  'schema_version',
  'source_metadata',
  'text'],
 'chars': 31502,
 'source_path': '../local/test_data/1.txt',
 'source_format': 'text',
 'backend': 'plain_text'}

{'document': '2.txt',
 'stage': 'preprocessed',
 'snapshot': '../local/snapshots/2/preprocessed.json',
 'top_level_keys': ['artifact_type',
  'preprocessing_metadata',
  'schema_version',
  'source_metadata',
  'text'],
 'chars': 53394,
 'source_path': '../local/test_data/2.txt',
 'source_format': 'text',
 'backend': 'plain_text'}

## Step 2: Extract

`run_extraction(PreprocessedDocument)` advances the object to `ExtractedDocument`. The snapshot is reloaded with `deserialize_extracted_document` before retrieval.

In [4]:
if not _should_run("extraction"):
    display("Skipping: phase {PHASE!r} < extraction")
else:
    for doc_path in DOC_PATHS:
        preprocessed = load_snapshot(doc_path, "preprocessed", deserialize_preprocessed_document)
        extraction_raw = run_extraction(preprocessed)
        extraction_payload = serialize_extracted_document(extraction_raw)
        extraction_path = write_snapshot(doc_path, "extraction", extraction_payload)
        extraction = load_snapshot(doc_path, "extraction", deserialize_extracted_document)

## Step 3: Infer

`infer_jurisdiction(ExtractedDocument)` produces the `InferredDocument`. The snapshot is reloaded with `deserialize_inferred_document` before retrieval.

This step is purely local — no I/O, no LLM, no rate limits.

In [5]:
if not _should_run("inferred"):
    display("Skipping: phase {PHASE!r} < inferred")
else:
    for doc_path in DOC_PATHS:
        extraction = load_snapshot(doc_path, "extraction", deserialize_extracted_document)
        inferred_raw = infer_jurisdiction(extraction)
        inferred_payload = serialize_inferred_document(inferred_raw)
        inferred_path = write_snapshot(doc_path, "inferred", inferred_payload)
        inferred = load_snapshot(doc_path, "inferred", deserialize_inferred_document)


## Step 4: Retrieve

`run_retrieval(InferredDocument)` advances the object to `RetrievedDocument`. The snapshot is reloaded with `deserialize_retrieved_document` before assessment.

This may call the configured CourtListener access service.

In [6]:
if not _should_run("retrieval"):
    display("Skipping: phase {PHASE!r} < retrieval")
else:
    for doc_path in DOC_PATHS:
        inferred = load_snapshot(doc_path, "inferred", deserialize_inferred_document)
        retrieval_raw = run_retrieval(inferred)
        retrieval_payload = serialize_retrieved_document(retrieval_raw)
        retrieval_path = write_snapshot(doc_path, "retrieval", retrieval_payload)
        retrieval = load_snapshot(doc_path, "retrieval", deserialize_retrieved_document)
        display({"document": doc_path.name, **summarize_snapshot("retrieval", retrieval_path, retrieval_payload)})
        display(retrieval_payload["retrievals"][:3])

'Skipping: phase {PHASE!r} < retrieval'

## Step 5: Assess

`run_assessment_async(RetrievedDocument)` advances the object to `AssessedDocument` and may call the configured LLM API in parallel. The snapshot is reloaded with `deserialize_assessed_document`.

In [7]:
import time

llm_config = llm_api_config_from_env(os.environ)
mellea_started: dict[tuple[str, str], float] = {}
active_document = ""


def on_mellea_call(ctx: MelleaCallContext) -> None:
    mellea_started[(active_document, ctx.citation_id)] = time.perf_counter()
    print(
        f"[mellea] start doc={active_document} id={ctx.citation_id} "
        f"extracted={ctx.extracted_case_name!r}",
        flush=True,
    )


def on_mellea_done(ctx: MelleaCallContext, item: CitationAssessmentResult) -> None:
    elapsed = time.perf_counter() - mellea_started[(active_document, ctx.citation_id)]
    case = item.case_name.initial.status.value
    followup = item.case_name.followup.status.value
    print(
        f"[mellea] done  doc={active_document} id={ctx.citation_id} "
        f"case={case} followup={followup} ({elapsed:.1f}s)",
        flush=True,
    )


if not _should_run("assessment"):
    display("Skipping: phase {PHASE!r} < assessment")
else:
    for doc_path in DOC_PATHS:
        active_document = doc_path.name
        retrieval = load_snapshot(doc_path, "retrieval", deserialize_retrieved_document)
        assessment_raw = await run_assessment_async(
            retrieval,
            mellea_concurrency=MELLEA_CONCURRENCY,
            on_mellea_call=on_mellea_call,
            on_mellea_done=on_mellea_done,
        )
        assessment_payload = serialize_assessed_document(assessment_raw)
        assessment_path = write_snapshot(doc_path, "assessment", assessment_payload)
        assessment = load_snapshot(doc_path, "assessment", deserialize_assessed_document)
        assessment_status_counts = {
            status: sum(item.status.value == status for item in assessment.assessments)
            for status in sorted({item.status.value for item in assessment.assessments})
        }
        display(
            {
                "document": doc_path.name,
                **summarize_snapshot("assessment", assessment_path, assessment_payload),
                "api_base": llm_config.api_base,
                "model": llm_config.model,
                "assessment_records": len(assessment.assessments),
                "assessment_status_counts": assessment_status_counts,
                "all_records_terminal": assessment.assessment_complete,
            }
        )

'Skipping: phase {PHASE!r} < assessment'